In [5]:
import os
import sys

import numpy as np
import pandas as pd
import psycopg2
from sqlalchemy import create_engine

from preprocessing import *

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
from features import *

DB_URL = (
"postgresql+psycopg://neondb_owner:npg_Bo2SUY6ngypR@"
    "ep-orange-frost-afcl94sd-pooler.c-2.us-west-2.aws.neon.tech/"
    "neondb?sslmode=require"

)

engine = create_engine(
        DB_URL,
        pool_pre_ping=True,
        pool_recycle=3600
        )

In [3]:
#Loading Data
df = pd.read_sql(
    """
    SELECT *
    FROM ml.fight_dataset
    """,
    engine
)

In [4]:
# Preprocess
df = preprocess_ranks(df)
df = preprocess_weight(df)

df = df.reset_index(drop=True)
df["fight_id"] = df.index

history = build_fighter_history(df)

df = add_fighter_cumulative_features(df, history)
df = add_striking_rolling_features(df, history)
df = add_grappling_rolling_features(df, history)

In [6]:
df = create_features(df)

In [9]:
df["fighter_1_win"] = (df["winner"] == df["fighter_1"]).astype(int)
df["fighter_1_win"].value_counts()

fighter_1_win
1    5519
0    3239
Name: count, dtype: int64